In [31]:
## IMPORT LIBRARIES ----------------------------------------------------------------------------------------------------
#https://github.com/nnc-ufmg/circadipy/blob/main/src/circadipy/analysis_examples/intellicage/intellicage_analysis.ipynb
%matplotlib qt

%reload_ext autoreload
%autoreload 3
import json
from circadipy import chrono_reader as chr  
import sys                                                                                                              # Import sys to add paths to libraries                                                                                                           # Import re to work with regular expressions
import glob                                                                                                             # Import glob to read files                                                                                                   # Import numpy to work with arrays and make calculations                                                                                            # Import time to measure time
import os                                                                                                               # Import path to work with paths                
import pandas as pd
from ranking_methods import build_all_proxies, evaluate_proxies
import numpy as np
from scipy import stats
from datetime import date, datetime, timedelta
from itertools import combinations
from scipy.stats import rankdata, spearmanr, kendalltau
from ranking_methods import rank_accuracy                                                                                   # Import pandas to work with dataframes
import warnings                                                                                                         # Import warnings to ignore warnings
warnings.filterwarnings('ignore')                                                                                       # Ignore warnings

## IMPORT CIRCADIPY ----------------------------------------------------------------------------------------------------

parent_path = os.path.dirname(os.path.dirname(os.getcwd()))
sys.path.append(parent_path)

## PCA Visualization - Dimensionality Reduction
from models import train_rf, train_gb, train_ridge, train_adaboost, train_extratrees, train_logistic
from summary import generate_summary_report
from util import generate_result, get_features, get_data_scaled, generate_features, generate_temporal_features, calculate_correlations, build_animal_protocols, get_sorted_animals_files, combine_all_features
from analysis_visualization import generate_methods_comparison, plot_animals_activity, plot_correlation, plot_pca_analysis, plot_cross_correlation, summary_visualization

In [22]:

best_feat = None   # None = auto-pick highest |ρ| feature from corr_df#
#best_feat = 'ibi_median'
#named_combo = ['ibi_median', 'hourly_entropy', 'power_8h', 'bout_len_cv']

named_combo = None
#named_combo = ['rhythm_ratio_24_over_harm', 'daily_peak_std', 'night_frac', 'num_bouts_night']

data_folder = "./data/dados_iniciais_estruturados"    
individual_files = glob.glob(data_folder + "/**/*.zip", recursive=True)
output_folder = './results/test_results_2026_new_data3'

# Build date range between two dates (inclusive)
start_date = date(2026, 3, 17)
end_date   = date(2026, 3, 20)
dates_to_keep = [(start_date + timedelta(days=i)).strftime("%Y-%m-%d")
                for i in range((end_date - start_date).days + 1)]
force_rewrite = False
individual_files = glob.glob(data_folder + "/**/*.txt", recursive=True)



if force_rewrite:
    for file in individual_files:
        folder = file.split("/")[-1].split(".zip")[0]

        
        root_folder = os.path.join(data_folder, folder)
        os.makedirs(root_folder, exist_ok=True)
        a = chr.intellicage_unwrapper([file], root_folder=root_folder, sampling_interval = '30T')
    


if dates_to_keep:
    
    individual_files = [f for f in individual_files if f.split("/")[-2].split("_")[0] in dates_to_keep]

ground_file = f'{data_folder}/ground.json'


if os.path.exists(ground_file):
    with open(ground_file, 'r') as f:
        ranks_ground = json.load(f)



In [23]:


#usando sum para concatenar os protocolos gera resultado melhor que o last
#Se eu quiser ler os protocolos eu preciso do zt_0_time, neste caso estou usando 20horas.
#Tambem precisariamos definir o labels dict, neste caso o que utilizariamos?
zt_0_time = 20  #Para gerar o actograma é usado o zt dado de quando a luz é acesa, que será as 20 horas. 
apply_filtering = True
labels_dict = {'cycle_types': ['LD'], 'test_labels': ['1_control_dl'], 'cycle_days': [1]}
animals = [int(k.split("_")[1]) for k in ranks_ground.keys()]
print(animals)
animals_files = get_sorted_animals_files(individual_files, animals)
print(f"Animals files: {animals_files}")
for k, v in animals_files.items():
    print(f"{k}: {v}")
animals_protocols, animals_by_day = build_animal_protocols(animals_files, apply_filtering=apply_filtering)




[6, 10, 11, 12, 7, 8, 9, 2, 1, 4, 3, 5]
Animal 1: 4
Animal 2: 4
Animal 3: 4
Animal 4: 4
Animal 5: 4
Animal 6: 4
Animal 7: 4
Animal 8: 4
Animal 9: 4
Animal 10: 4
Animal 11: 4
Animal 12: 4
Animals files: {1: ['./data/dados_iniciais_estruturados/2026-03-17_18/animal_1.txt', './data/dados_iniciais_estruturados/2026-03-18_17/animal_1.txt', './data/dados_iniciais_estruturados/2026-03-19_17/animal_1.txt', './data/dados_iniciais_estruturados/2026-03-20_18/animal_1.txt'], 2: ['./data/dados_iniciais_estruturados/2026-03-17_18/animal_2.txt', './data/dados_iniciais_estruturados/2026-03-18_17/animal_2.txt', './data/dados_iniciais_estruturados/2026-03-19_17/animal_2.txt', './data/dados_iniciais_estruturados/2026-03-20_18/animal_2.txt'], 3: ['./data/dados_iniciais_estruturados/2026-03-17_18/animal_3.txt', './data/dados_iniciais_estruturados/2026-03-18_17/animal_3.txt', './data/dados_iniciais_estruturados/2026-03-19_17/animal_3.txt', './data/dados_iniciais_estruturados/2026-03-20_18/animal_3.txt'], 4:

In [24]:
all_features, feature_cols, X_scaled, ground = get_features(animals_protocols, ranks_ground, output_folder)
all_features['actual_rank'] = ground

output_correlation = f'{output_folder}/feature_correlations.csv'
corr_df = calculate_correlations(all_features, feature_cols, ground, output_path=output_correlation)

Saving features on ./results/test_results_2026_new_data3/basic_features.csv
Saving temporal features on ./results/test_results_2026_new_data3/temporal_features.csv
Saving all features on ./results/test_results_2026_new_data3/all_features.csv
Saving correlation on ./results/test_results_2026_new_data3/feature_correlations.csv


In [25]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Diagnostic: feature heatmap sorted by true rank ─────────────────────────
# Shows z-scored feature values per animal (rows = animals sorted by rank 1→N)
# Features sorted by |ρ| with actual_rank. Ideal: a clear gradient top→bottom.

top_diag = 30   # how many top features to show
diag_feats = corr_df[corr_df.index.isin(feature_cols)].head(top_diag).index.tolist()

# Sort animals by actual rank
af_sorted = all_features.sort_values('actual_rank')
animal_labels = [f"{a} (#{int(r)})" for a, r in zip(af_sorted['animal'], af_sorted['actual_rank'])]

# Z-score matrix
from sklearn.preprocessing import StandardScaler
Z = StandardScaler().fit_transform(af_sorted[diag_feats].values)

# Sign-align so that positive z → higher rank (consistent visual direction)
for j, feat in enumerate(diag_feats):
    rho_sign = np.sign(corr_df.loc[feat, 'correlation'])
    if rho_sign < 0:
        Z[:, j] *= -1

fig_heat = go.Figure(go.Heatmap(
    z=Z,
    x=diag_feats,
    y=animal_labels,
    colorscale='RdBu',
    zmid=0,
    colorbar=dict(title='z (sign-aligned)'),
))
fig_heat.update_layout(
    title='Feature Heatmap — Animals sorted by Rank (top = most dominant)',
    height=max(400, 40 * len(af_sorted)),
    width=max(900, 30 * len(diag_feats)),
    xaxis=dict(tickangle=-45, tickfont=dict(size=9)),
    yaxis=dict(tickfont=dict(size=9)),
    plot_bgcolor='white', paper_bgcolor='white',
)
fig_heat.write_html(f"{output_folder}/feature_heatmap.html")
fig_heat.show()

# ── Scatter grid: top-6 features vs actual rank ──────────────────────────────
top6 = diag_feats[:6]
fig_scatter = make_subplots(rows=2, cols=3,
    subplot_titles=[f"{f}<br>ρ={corr_df.loc[f,'correlation']:+.3f}" for f in top6])

for idx, feat in enumerate(top6):
    r, c = divmod(idx, 3)
    x_vals = af_sorted['actual_rank'].tolist()
    y_vals = af_sorted[feat].tolist()
    fig_scatter.add_trace(go.Scatter(
        x=x_vals, y=y_vals, mode='markers+text',
        text=af_sorted['animal'].tolist(),
        textposition='top center', textfont=dict(size=7),
        marker=dict(size=9, color=x_vals, colorscale='Viridis', showscale=False),
        showlegend=False,
    ), row=r+1, col=c+1)
    fig_scatter.update_xaxes(title_text='True rank', row=r+1, col=c+1)
    fig_scatter.update_yaxes(title_text=feat, row=r+1, col=c+1)

fig_scatter.update_layout(
    title='Top Feature Scatter vs True Rank',
    height=600, width=1100,
    plot_bgcolor='white', paper_bgcolor='white',
)
fig_scatter.write_html(f"{output_folder}/feature_scatter_vs_rank.html")
fig_scatter.show()
print(f"Saved heatmap → {output_folder}/feature_heatmap.html")
print(f"Saved scatter → {output_folder}/feature_scatter_vs_rank.html")


Saved heatmap → ./results/test_results_2026_new_data3/feature_heatmap.html
Saved scatter → ./results/test_results_2026_new_data3/feature_scatter_vs_rank.html


In [26]:

# ── Build TOP_FEATURES dynamically from corr_df (cell 4 output) ─────────────
min_rho  = 0.20   # lowered threshold — cast wider net, search will still rank by |ρ|
top_n    = 30     # more candidates

top_corr = corr_df[corr_df.index.isin(feature_cols)].copy()
top_corr = top_corr[top_corr['correlation'].abs() >= min_rho].head(top_n)

TOP_FEATURES = {
    feat: (int(np.sign(row['correlation'])), round(abs(row['correlation']), 3))
    for feat, row in top_corr.iterrows()
}

print(f"TOP_FEATURES built from corr_df ({len(TOP_FEATURES)} features, |ρ| >= {min_rho}):")
print(f"{'Feature':<25} {'sign':>5}  {'|ρ|':>6}  {'p':>7}")
print("-" * 55)
for feat, (sign, rho_abs) in TOP_FEATURES.items():
    p = corr_df.loc[feat, 'p_value']
    sig = "**" if p < 0.01 else " *" if p < 0.05 else " ~" if p < 0.1 else ""
    print(f"  {feat:<23} {sign:>+5}  {rho_abs:>6.3f}  {p:>7.3f} {sig}")

# ── Exhaustive search ────────────────────────────────────────────────────────
available_feats = {f: v for f, v in TOP_FEATURES.items() if f in feature_cols}
feat_names = list(available_feats.keys())
y_true_cmp = all_features['actual_rank'].values

feat_vectors = {}
for feat, (sign, weight) in available_feats.items():
    col_idx = feature_cols.index(feat)
    feat_vectors[feat] = sign * X_scaled[:, col_idx]

best_results = []
total_combos = sum(len(list(combinations(feat_names, k))) for k in range(1, 6)) * 2

for k in range(1, 6):
    for combo in combinations(feat_names, k):
        composite_c = sum(feat_vectors[f] for f in combo)
        for polarity in [1, -1]:
            pred = rankdata(-polarity * composite_c).astype(int)
            rho_c, p_c = spearmanr(y_true_cmp, pred)
            mae_c = np.abs(y_true_cmp - pred).mean()
            if rho_c > 0:
                best_results.append((rho_c, p_c, mae_c, k, combo, polarity))

best_results.sort(key=lambda x: (-x[0], x[2]))

print(f"\nTop 20 feature combinations (out of {len(best_results)} valid, {total_combos} tested):\n")
print(f"{'ρ':>6} {'p':>11} {'MAE':>5}  {'k':>2}  Features")
print("-" * 90)
for rho_c, p_c, mae_c, k, combo, pol in best_results[:20]:
    sig = "**" if p_c < 0.01 else " *" if p_c < 0.05 else " ~" if p_c < 0.1 else "  "
    print(f"{rho_c:+.3f} {p_c:11.3e} {mae_c:5.2f}  {k:2d}  {', '.join(combo)} {sig}")

# ── Feature vote across top-10 results ──────────────────────────────────────
top_feats_pool = {}
for rho_c, p_c, mae_c, k, combo, pol in best_results[:10]:
    for f in combo:
        top_feats_pool[f] = top_feats_pool.get(f, 0) + rho_c

top_feats_ranked = sorted(top_feats_pool.items(), key=lambda x: -x[1])
print(f"\nMost frequent features in top-10 combos (by accumulated ρ):")
for feat, score in top_feats_ranked:
    sign, _ = available_feats[feat]
    print(f"  {feat:<25s}  score={score:.3f}  sign={'↑' if sign>0 else '↓'}")

# ── Apply the best combination ──────────────────────────────────────────────
best_rho, best_p, best_mae, best_k, best_combo, best_pol = best_results[0]
best_combo_str = " + ".join(best_combo)

print(f"\n→ Best combo (k={best_k}): {list(best_combo)}")
print(f"  ρ={best_rho:+.3f}, p={best_p:.3e}, MAE={best_mae:.2f}")

composite_raw = best_pol * sum(feat_vectors[f] for f in best_combo)
composite_oriented = -composite_raw  # menor score = maior dominância
pred_best = rankdata(composite_oriented).astype(int)
all_features['predicted_rank_best'] = pred_best
all_features['composite_score_best'] = composite_oriented

comparison_best = all_features.set_index('animal')[['predicted_rank_best', 'composite_score_best']].copy()
comparison_best['true_rank'] = pd.Series(ranks_ground)
comparison_best = comparison_best.sort_values('true_rank')
comparison_best['error'] = (comparison_best['predicted_rank_best'] - comparison_best['true_rank']).abs()

n = len(comparison_best)
print(f"\n  Spearman ρ      : {best_rho:+.3f}  (p={best_p:.3e})")
print(f"  MAE             : {best_mae:.2f} ranks")
print(f"  Exact match     : {(comparison_best['error']==0).sum()}/{n}  ({100*(comparison_best['error']==0).mean():.0f}%)")
print(f"  Within ±1 rank  : {(comparison_best['error']<=1).sum()}/{n}  ({100*(comparison_best['error']<=1).mean():.0f}%)")
print(f"  Within ±2 ranks : {(comparison_best['error']<=2).sum()}/{n}  ({100*(comparison_best['error']<=2).mean():.0f}%)")
print(f"\n{'Animal':<12} {'True':>6} {'Pred':>6} {'Error':>6}")
print("-" * 34)
for animal, row in comparison_best.iterrows():
    flag = " ✓" if row['error'] == 0 else (f" ~{int(row['error'])}" if row['error'] <= 2 else f" ✗{int(row['error'])}")
    print(f"{animal:<12} {int(row['true_rank']):>6} {int(row['predicted_rank_best']):>6} {int(row['error']):>6}{flag}")

# ── Save ranked output (original animal order) ───────────────────────────────
ranked_output = comparison_best.copy().reset_index()
ranked_output = ranked_output.rename(columns={
    'predicted_rank_best': 'predicted_rank',
    'composite_score_best': 'composite_score',
    'error': 'absolute_error',
})
ranked_output['features_used'] = best_combo_str
ranked_output['spearman_rho']  = round(best_rho, 4)
ranked_output['p_value']       = f"{best_p:.3e}"
ranked_output['mae']           = round(best_mae, 4)

ranked_output_path = f"{output_folder}/best_combo_ranked.csv"
ranked_output.to_csv(ranked_output_path, index=False)

ranked_output_path = f"./best_combo_ranked.csv"
ranked_output.to_csv(ranked_output_path, index=False)

print(f"\n── Best combo ranked output (original animal order) ──")
print(ranked_output[['animal', 'predicted_rank', 'true_rank', 'absolute_error', 'composite_score']].to_string(index=False))
print(f"\nSaved to {ranked_output_path}")

# ── Save single-feature output (same layout as best combo) ───────────────────
single_feature_name = 'rhythm_ratio_24_over_harm'
if single_feature_name not in feat_vectors:
    raise ValueError(f"Single feature '{single_feature_name}' is not available in selected candidate features.")

single_candidates = [r for r in best_results if len(r[4]) == 1 and r[4][0] == single_feature_name]
if not single_candidates:
    raise ValueError(f"Single feature '{single_feature_name}' was not found among valid combinations (rho > 0).")

single_rho, single_p, single_mae, _, single_combo, single_pol = sorted(
    single_candidates, key=lambda x: (-x[0], x[2])
)[0]
single_score_oriented = -single_pol * feat_vectors[single_feature_name]  # menor score = maior dominância
single_pred = rankdata(single_score_oriented).astype(int)

single_df = pd.DataFrame({
    'animal': all_features['animal'].tolist(),
    'predicted_rank': single_pred.astype(int),
    'composite_score': single_score_oriented.astype(float),
    'true_rank': y_true_cmp.astype(int),
})
single_df['absolute_error'] = (single_df['predicted_rank'] - single_df['true_rank']).abs()
single_df['features_used'] = single_feature_name
single_df['spearman_rho'] = round(float(single_rho), 4)
single_df['p_value'] = f"{float(single_p):.3e}"
single_df['mae'] = round(float(single_mae), 4)
best_combo_order_animals = ranked_output['animal'].tolist()
single_df = single_df.set_index('animal').loc[best_combo_order_animals].reset_index()

ranked_output_path = f"{output_folder}/best_combo_ranked.csv"
ranked_output.to_csv(ranked_output_path, index=False)
ranked_output_path = f"./best_combo_ranked.csv"
ranked_output.to_csv(ranked_output_path, index=False)

single_output_path = f"{output_folder}/single_feature_ranked.csv"
single_df.to_csv(single_output_path, index=False)
single_output_path = f"./single_feature_ranked.csv"
single_df.to_csv(single_output_path, index=False)

print(f"\n── Single feature ranked output ({single_feature_name}) ──")
print(single_df[['animal', 'predicted_rank', 'true_rank', 'absolute_error', 'composite_score']].to_string(index=False))
print(f"\nSaved to {single_output_path}")

# ── Save top-5 combos placement/prediction ───────────────────────────────────
top_k_to_save = min(5, len(best_results))
top5_ranked_rows = []
top5_summary_rows = []

for pos, (rho_c, p_c, mae_c, k_c, combo_c, pol_c) in enumerate(best_results[:top_k_to_save], start=1):
    combo_str = " + ".join(combo_c)
    combo_raw = pol_c * sum(feat_vectors[f] for f in combo_c)
    combo_oriented = -combo_raw  # menor score = maior dominância
    combo_pred = rankdata(combo_oriented).astype(int)

    combo_df = pd.DataFrame({
        'animal': all_features['animal'].tolist(),
        'true_rank': y_true_cmp.astype(int),
        'predicted_rank': combo_pred.astype(int),
        'composite_score': combo_oriented.astype(float),
    })
    combo_df['absolute_error'] = (combo_df['predicted_rank'] - combo_df['true_rank']).abs()
    combo_df['combo_position'] = pos
    combo_df['features_used'] = combo_str
    combo_df['spearman_rho'] = round(float(rho_c), 4)
    combo_df['p_value'] = f"{float(p_c):.3e}"
    combo_df['mae'] = round(float(mae_c), 4)
    top5_ranked_rows.append(combo_df)

    top5_summary_rows.append({
        'combo_position': pos,
        'features_used': combo_str,
        'k': int(k_c),
        'spearman_rho': round(float(rho_c), 4),
        'p_value': f"{float(p_c):.3e}",
        'mae': round(float(mae_c), 4),
    })

top5_ranked_output = pd.concat(top5_ranked_rows, ignore_index=True)
top5_summary_output = pd.DataFrame(top5_summary_rows)
best_rho_ref = float(top5_summary_output.loc[top5_summary_output['combo_position'] == 1, 'spearman_rho'].iloc[0])
best_mae_ref = float(top5_summary_output.loc[top5_summary_output['combo_position'] == 1, 'mae'].iloc[0])
top5_summary_output['delta_spearman_rho'] = (top5_summary_output['spearman_rho'] - best_rho_ref).round(4)
top5_summary_output['delta_mae'] = (top5_summary_output['mae'] - best_mae_ref).round(4)

top5_ranked_path = f"{output_folder}/top5_combos_ranked.csv"
top5_summary_path = f"{output_folder}/top5_combos_summary.csv"
top5_ranked_output.to_csv(top5_ranked_path, index=False)
top5_summary_output.to_csv(top5_summary_path, index=False)

print(f"Saved top-5 ranked output to {top5_ranked_path}")
print(f"Saved top-5 summary to {top5_summary_path}")

feature_rhos = corr_df.copy()
feature_rhos.rename(columns={'correlation': 'rho'}, inplace=True)
feature_rhos = feature_rhos.reset_index(names="feature")
feature_rhos.drop(columns=['p_value'], inplace=True)
feature_rhos = feature_rhos.set_index("feature")["rho"]
feature_rhos.to_csv("./features_rhos.csv")
feature_rhos.head()


TOP_FEATURES built from corr_df (30 features, |ρ| >= 0.2):
Feature                    sign     |ρ|        p
-------------------------------------------------------
  rhythm_ratio_24_over_harm    +1   0.790    0.002 **
  power_24h                  +1   0.706    0.010  *
  autocorr_12h               -1   0.664    0.018  *
  power_12h                  -1   0.657    0.020  *
  cosinor_amplitude          +1   0.657    0.020  *
  median_activity            -1   0.647    0.023  *
  activity_kurtosis          -1   0.608    0.036  *
  high_activity_frac         +1   0.578    0.049  *
  activity_skew              -1   0.559    0.059  ~
  bout_len_cv                -1   0.545    0.067  ~
  max_activity               -1   0.545    0.067  ~
  peak_to_mean_ratio         -1   0.524    0.080  ~
  max_median_ratio           +1   0.517    0.085  ~
  cosinor_mesor              +1   0.517    0.085  ~
  night_mean_activity        -1   0.476    0.118 
  night_activity             -1   0.476    0.118 
  roll

feature
rhythm_ratio_24_over_harm    0.790210
power_24h                    0.706294
autocorr_12h                -0.664336
power_12h                   -0.657343
cosinor_amplitude            0.657343
Name: rho, dtype: float64

In [32]:
output_path = "./pred_data_to_predict_estruturado2.csv"
generate_result(
    all_features, feature_cols, X_scaled, ground, feature_rhos, output_path  
)

Using feature_rhos provided externally (e.g. from a reference dataset).
Using feature_rhos provided externally (e.g. from a reference dataset).
Using feature_rhos provided externally (e.g. from a reference dataset).
Using feature_rhos provided externally (e.g. from a reference dataset).
Using feature_rhos provided externally (e.g. from a reference dataset).
Using feature_rhos provided externally (e.g. from a reference dataset).


##### Total activy per day tem a soma de todos os dias. Ë possivel ver isso claramente abrindo o arquivo collect data onde será visto a atividade por zt de 0.5 em 0.5, somando-se todos os valores da coluna o valor para cada dia será igual ao total day activity